In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from OTTXDM import new_centroids

df = pd.read_csv('iris.csv')
df = df.dropna().reset_index(drop=True)

target_col = df.columns[-1]  # 0 neu class o dau
X_df = df.drop(columns=[target_col]).copy()
y = df[target_col].values

# y_raw = df[target_col].values
# le = LabelEncoder()
# y = le.fit_transform(y_raw)

X_new_df = pd.DataFrame([[6, 3, 0.7, 0.9]], columns = X_df.columns)

for col in X_df.columns:
    if X_df[col].dtype == 'object':
        enc = LabelEncoder()
        X_df = enc.fit_transform(X_df[col])
        X_new_df[col] = enc.transform(X_new_df[col])

X_raw = X_df.values.astype(float)
X_new_raw = X_new_df.values.astype(float)

a,b = 0, 10

scaler = MinMaxScaler(feature_range=(a,b))
X = scaler.fit_transform(X_raw)
X_new = scaler.transform(X_new_raw)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

print(f"{len(X)}, {X.shape[1]}, {np.unique(y)}")



150, 4, ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


In [4]:
def gaussian(x, mean, var):
    eps = 1e-9
    return ( 1.0 / np.sqrt( 2.0 * np.pi * ( var + eps ))) * np.exp(-(( x - mean ) ** 2 / ( 2 * (var + eps))))

classes = np.unique(y)
probs = {}

for c in classes:
    X_c = X[y == c]
    prior = len(X_c) / len(X)
    mean, var = X_c.mean(axis = 0), X_c.var(axis = 0)
    likelihood = np.prod(gaussian(X_new[0], mean, var))
    probs[c] = prior * likelihood

total = sum(probs.values()) + 1e-15

for c in classes:
    probs[c] /= total

pred_c = max(probs, key = probs.get)
print("X thuoc lop:", pred_c)

X thuoc lop: Iris-setosa


In [6]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train, y_train)
print("Xac suat:", np.round(nb.predict_proba(X_new)[0], 2))
print("X thuoc lop:", nb.predict(X_new)[0])

Xac suat: [0.44 0.56 0.  ]
X thuoc lop: Iris-versicolor


In [8]:
K = 2
np.random.seed(42)

centroids = X[np.random.choice(len(X), K, replace = False)]

for _ in range(100):
    distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis = 2)
    labels = np.argmin(distances, axis = 1)

    new_centroids = np.array([
        X[labels == k].mean(axis = 0) if np.any(labels == k )else centroids[k] for k in range(K)
    ])

    if np.allclose(centroids, new_centroids):
        break
    centroids = new_centroids

for k in range(K):
    print(f"Cum {k}: {np.sum(labels == k)} mau | Tam: {np.round(centroids[k], 3)}")

print("X thuoc cum: ", np.argmin(np.linalg.norm(centroids - X_new[0], axis = 1)))

Cum 0: 100 mau | Tam: [5.45  3.633 6.62  6.567]
Cum 1: 50 mau | Tam: [1.961 5.908 0.786 0.6  ]
X thuoc cum:  1


In [10]:
from sklearn.cluster import KMeans

km = KMeans( n_clusters = K, random_state = 42, n_init = 10)
labels_lib = km.fit(X)

for k in range(K):
    print(f"Cum {k}: {np.sum(labels_lib ==k)} mau | Tam: {np.round(km.cluster_centers_[k], 3)}")

print("X thuoc cum: ", km.predict(X_new)[0])

Cum 0: 0 mau | Tam: [5.45  3.633 6.62  6.567]
Cum 1: 0 mau | Tam: [1.961 5.908 0.786 0.6  ]
X thuoc cum:  1


In [18]:
K = 3
clusters = [[i] for i in range(len(X))]
centers = [X[i] for i in range(len(X))]

while len(clusters) > K:
    min_dist, pair = float('inf'), (0, 1)
    for i in range(len(centers)):
        for j in range( i+1, len(centers)):
            n1, n2 = len(clusters[i]), len(clusters[j])
            d = (n1 * n2 / (n1 + n2)) * np.sum((centers[i] - centers[j]) ** 2)
            if d < min_dist:
                min_dist, pair = d, (i, j)

    i, j = pair
    clusters[i].extend(clusters[j])
    centers[i] = X[clusters[i]].mean(axis = 0)
    clusters.pop(j)
    centers.pop(j)

for idx, c in enumerate(clusters):
    print(f"Cum {idx}: {len(c)} mau")

print("X thuoc cum:", np.argmin([np.linalg.norm(c - X_new[0]) for c in centers]))

Cum 0: 50 mau
Cum 1: 67 mau
Cum 2: 33 mau
X thuoc cum: 0


In [17]:
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(n_clusters = K, linkage = 'ward')
labels = agg.fit_predict(X)

centers_lib = []
for k in range(K):
    print(f"Cum {k}: {np.sum(labels == k)} man")
    centers_lib.append(X[labels == k].mean(axis = 0))

print("X thuoc cum: ", np.argmin([np.linalg.norm(c - X_new[0]) for c in centers_lib]))

Cum 0: 67 man
Cum 1: 50 man
Cum 2: 33 man
X thuoc cum:  1
